# 0. Setup
FinGuard loan default risk pipeline. CPU-only. Production target: <200ms single-row inference, 1x .pkl artifact.

In [1]:
# Install once (Colab/fresh CPU env). Comment out if already installed.
# !pip install -q xgboost lightgbm catboost shap scikit-learn pandas numpy joblib

import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import shap

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_FOLDS = 5
FP_COST = 5   # business cost weight: false positive (wrongly flag good payer) costs 5x a false negative
FN_COST = 1
np.random.seed(RANDOM_STATE)


# 1. Load Data
Placeholder paths for Home Credit files. Every load is try/except-wrapped -- the pipeline must run end-to-end even if only application_train.csv is present, since production inference only ever sees application-level fields.

In [2]:
from pathlib import Path

# Resolve DATA_DIR robustly: supports running from repo root or from notebook dir.
# Actual file on disk is HC_application_train.csv (166 MB) inside "cred default testing jupyter/"
CANDIDATES = [
    Path("./cred default testing jupyter"),
    Path("cred default testing jupyter"),
    Path("./"),
    Path(".") / "cred default testing jupyter",
    Path.cwd() / "cred default testing jupyter",
]
DATA_DIR = None
for cand in CANDIDATES:
    if (cand / "HC_application_train.csv").exists() or (cand / "application_train.csv").exists():
        DATA_DIR = cand
        break
if DATA_DIR is None:
    DATA_DIR = Path("./")
DATA_DIR = str(DATA_DIR).replace("\\", "/").rstrip("/") + "/"
print(f"[INFO] DATA_DIR resolved to: {DATA_DIR}")

DTYPE_MAP = {
    'SK_ID_CURR': 'int32', 'TARGET': 'int8', 'CNT_CHILDREN': 'int8',
    'AMT_INCOME_TOTAL': 'float32', 'AMT_CREDIT': 'float32',
    'AMT_ANNUITY': 'float32', 'AMT_GOODS_PRICE': 'float32',
    'REGION_POPULATION_RELATIVE': 'float32', 'DAYS_BIRTH': 'int32',
    'DAYS_EMPLOYED': 'int32', 'DAYS_REGISTRATION': 'float32',
    'DAYS_ID_PUBLISH': 'int32', 'CNT_FAM_MEMBERS': 'float32',
    'EXT_SOURCE_1': 'float32', 'EXT_SOURCE_2': 'float32', 'EXT_SOURCE_3': 'float32',
    'OWN_CAR_AGE': 'float32', 'REGION_RATING_CLIENT': 'int8',
}

def resolve_app_train_path():
    for name in ["HC_application_train.csv", "application_train.csv"]:
        p = Path(DATA_DIR) / name
        if p.exists():
            return str(p)
    return str(Path(DATA_DIR) / "application_train.csv")

def safe_read_csv(path, **kwargs):
    try:
        df = pd.read_csv(path, **kwargs)
        print(f"Loaded {path}: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"[WARN] {path} not found - skipping, dependent features will be NaN-filled.")
        return None
    except Exception as e:
        print(f"[WARN] Failed to load {path}: {e} - skipping.")
        return None

app_train_path = resolve_app_train_path()
print(f"[INFO] Loading primary table from: {app_train_path}")
app_train = safe_read_csv(app_train_path, dtype=DTYPE_MAP, low_memory=False)
assert app_train is not None, "application_train.csv is mandatory - cannot proceed without primary table."
assert app_train['SK_ID_CURR'].duplicated().sum() == 0, "Duplicate SK_ID_CURR - primary key violated."

bureau = safe_read_csv(f"{DATA_DIR}bureau.csv")
bureau_balance = safe_read_csv(f"{DATA_DIR}bureau_balance.csv")
prev_app = safe_read_csv(f"{DATA_DIR}previous_application.csv")
installments = safe_read_csv(f"{DATA_DIR}installments_payments.csv")
credit_card = safe_read_csv(f"{DATA_DIR}credit_card_balance.csv")
pos_cash = safe_read_csv(f"{DATA_DIR}POS_CASH_balance.csv")

FINGUARD_FIELDS = [c for c in app_train.columns]
print(f"application_train shape: {app_train.shape}, TARGET rate: {app_train['TARGET'].mean():.4f}")


[INFO] DATA_DIR resolved to: ./
[INFO] Loading primary table from: HC_application_train.csv
Loaded HC_application_train.csv: (307511, 122)
[WARN] ./bureau.csv not found - skipping, dependent features will be NaN-filled.
[WARN] ./bureau_balance.csv not found - skipping, dependent features will be NaN-filled.
[WARN] ./previous_application.csv not found - skipping, dependent features will be NaN-filled.
[WARN] ./installments_payments.csv not found - skipping, dependent features will be NaN-filled.
[WARN] ./credit_card_balance.csv not found - skipping, dependent features will be NaN-filled.
[WARN] ./POS_CASH_balance.csv not found - skipping, dependent features will be NaN-filled.
application_train shape: (307511, 122), TARGET rate: 0.0807


# 2. Cleaning Functions

In [3]:
def clean_application(df):
    df = df.copy()
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype('int8')
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    for col in ['DAYS_BIRTH', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']:
        if col in df.columns:
            df[col] = df[col].abs()
    for col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT']:
        if col in df.columns:
            q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
            iqr = q3 - q1
            lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
            df[col] = df[col].clip(lo, hi)
    return df

def drop_low_value_columns(df, finguard_schema):
    missing_frac = df.isnull().mean()
    high_null_cols = missing_frac[missing_frac > 0.60].index.tolist()
    droppable = [c for c in high_null_cols if c not in finguard_schema and c not in ('TARGET', 'SK_ID_CURR')]
    nunique = df.nunique(dropna=False)
    zero_var = nunique[nunique <= 1].index.tolist()
    to_drop = list(set(droppable + zero_var))
    print(f"Dropping {len(to_drop)} columns (high-null & non-FinGuard, or zero-variance): {to_drop[:15]}...")
    return df.drop(columns=to_drop, errors='ignore'), to_drop


# 3. Feature Engineering Functions

In [4]:
def engineer_application_features(df):
    """All ratios computed from application-level fields only -- this function is what production
    inference calls, so nothing here may depend on aux tables."""
    df = df.copy()
    eps = 1e-6

    df['AGE_YEARS'] = df['DAYS_BIRTH'] / 365
    df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'] / 365
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + eps)
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['CREDIT_TERM_YEARS'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + eps)
    df['EMPLOYED_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['DAYS_EMPLOYED_PERCENT'] = df['YEARS_EMPLOYED'] / (df['AGE_YEARS'] + eps)
    df['INCOME_PER_FAMILY'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, 1)
    df['INCOME_PER_CHILD'] = df['AMT_INCOME_TOTAL'] / (1 + df['CNT_CHILDREN'])
    df['CHILDREN_RATIO'] = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS'].replace(0, 1)
    if 'OWN_CAR_AGE' in df.columns:
        df['CAR_AGE_RATIO'] = df['OWN_CAR_AGE'] / (df['AGE_YEARS'] + eps)
    if 'DAYS_REGISTRATION' in df.columns:
        df['REGISTRATION_AGE_RATIO'] = df['DAYS_REGISTRATION'] / (df['DAYS_BIRTH'] + eps)
    if 'DAYS_ID_PUBLISH' in df.columns:
        df['ID_PUBLISH_AGE_RATIO'] = df['DAYS_ID_PUBLISH'] / (df['DAYS_BIRTH'] + eps)

    # EXT_SOURCE composites -- highest-signal block in every reference solution
    ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]
    if ext_cols:
        df['EXT_SOURCE_MEAN'] = df[ext_cols].mean(axis=1)
        df['EXT_SOURCE_STD'] = df[ext_cols].std(axis=1)
        df['EXT_SOURCE_MIN'] = df[ext_cols].min(axis=1)
        df['EXT_SOURCE_MAX'] = df[ext_cols].max(axis=1)
        df['EXT_SOURCE_NANCOUNT'] = df[ext_cols].isnull().sum(axis=1)
        df['EXT_SOURCE_PRODUCT'] = df[ext_cols].fillna(1).prod(axis=1)


        # Weighted EXT_SOURCE (weights from permutation importance per ML plan)
        df['EXT_SOURCE_1_2_WEIGHTED'] = 0.5*df.get('EXT_SOURCE_1', 0) + 0.3*df.get('EXT_SOURCE_2', 0) + 0.2*df.get('EXT_SOURCE_3', 0)
    # Document / contact aggregates
    doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
    if doc_cols:
        df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)
    contact_cols = [c for c in ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
                                 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL'] if c in df.columns]
    if contact_cols:
        df['CONTACT_FLAG_SUM'] = df[contact_cols].sum(axis=1)

    if 'OBS_30_CNT_SOCIAL_CIRCLE' in df.columns and 'DEF_30_CNT_SOCIAL_CIRCLE' in df.columns:
        df['SOCIAL_CIRCLE_DEF_RATIO'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] / (1 + df['OBS_30_CNT_SOCIAL_CIRCLE'])

    return df


def aggregate_bureau(bureau, bureau_balance):
    """Placeholder-safe. Returns None if bureau.csv absent -> caller skips the merge entirely."""
    if bureau is None:
        return None
    try:
        b = bureau.copy()
        agg = b.groupby('SK_ID_CURR').agg(
            BUREAU_LOANS_COUNT=('SK_ID_BUREAU', 'count'),
            BUREAU_ACTIVE_LOANS_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
            BUREAU_DEFAULT_AVG=('CREDIT_DAY_OVERDUE', lambda x: (x > 0).mean()),
            BUREAU_CREDIT_SUM_TOTAL=('AMT_CREDIT_SUM', 'sum'),
            BUREAU_CREDIT_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
            BUREAU_DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
        ).reset_index()
        agg['BUREAU_CREDIT_SUM_DEBT_RATIO'] = agg['BUREAU_CREDIT_SUM_DEBT'] / (agg['BUREAU_CREDIT_SUM_TOTAL'] + 1e-6)
        return agg
    except Exception as e:
        print(f"[WARN] bureau aggregation failed: {e} -- skipping bureau features.")
        return None


def aggregate_previous_application(prev_app):
    if prev_app is None:
        return None
    try:
        p = prev_app.copy()
        agg = p.groupby('SK_ID_CURR').agg(
            PREV_APP_COUNT=('SK_ID_PREV', 'count'),
            PREV_APPROVED_RATIO=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').mean()),
            PREV_REFUSED_RATIO=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').mean()),
            PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
        ).reset_index()
        return agg
    except Exception as e:
        print(f"[WARN] previous_application aggregation failed: {e} -- skipping.")
        return None


def aggregate_installments(installments):
    if installments is None:
        return None
    try:
        i = installments.copy()
        i['LATE'] = (i['DAYS_ENTRY_PAYMENT'] > i['DAYS_INSTALMENT']).astype('int8')
        i['PAYMENT_DIFF'] = i['AMT_INSTALMENT'] - i['AMT_PAYMENT']
        agg = i.groupby('SK_ID_CURR').agg(
            MISSED_PAYMENT_COUNT=('LATE', 'sum'),
            LATE_PAYMENT_RATIO=('LATE', 'mean'),
            PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
        ).reset_index()
        return agg
    except Exception as e:
        print(f"[WARN] installments aggregation failed: {e} -- skipping.")
        return None


def aggregate_credit_card(credit_card):
    if credit_card is None:
        return None
    try:
        c = credit_card.copy()
        c['UTILIZATION'] = c['AMT_BALANCE'] / (c['AMT_CREDIT_LIMIT_ACTUAL'] + 1e-6)
        agg = c.groupby('SK_ID_CURR').agg(
            CC_UTILIZATION_MEAN=('UTILIZATION', 'mean'),
            CC_LATE_PAYMENT_COUNT=('SK_DPD', lambda x: (x > 0).sum()),
        ).reset_index()
        return agg
    except Exception as e:
        print(f"[WARN] credit_card aggregation failed: {e} -- skipping.")
        return None


def aggregate_pos_cash(pos_cash):
    if pos_cash is None:
        return None
    try:
        pc = pos_cash.copy()
        agg = pc.groupby('SK_ID_CURR').agg(
            POS_DPD_MEAN=('SK_DPD', 'mean'),
            POS_COMPLETED_RATIO=('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').mean()),
        ).reset_index()
        return agg
    except Exception as e:
        print(f"[WARN] POS_CASH aggregation failed: {e} -- skipping.")
        return None


def merge_aux_features(df, bureau_agg, prev_agg, install_agg, cc_agg, pos_agg):
    """Left-merge every available aux aggregate. Missing aux tables simply leave NaNs,
    which the imputer handles downstream -- no hard dependency on any aux file."""
    df = df.copy()
    for agg in [bureau_agg, prev_agg, install_agg, cc_agg, pos_agg]:
        if agg is not None:
            df = df.merge(agg, on='SK_ID_CURR', how='left')
    return df


# 4. Encoding + Split
K-fold target encoding for high-cardinality categoricals -- fit inside each training fold only, never on the full dataset before CV. This is the #1 leakage bug in naive Kaggle scripts (encoding on full data before split inflates CV AUC and doesn't generalize).

In [5]:
HIGH_CARD_COLS = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
TARGET_ENC_SMOOTHING = 20

def kfold_target_encode(train_df, val_df, col, target_col='TARGET', smoothing=TARGET_ENC_SMOOTHING):
    """Fit encoding on train_df only, apply to both train_df and val_df. Caller must pass
    fold-local train/val splits -- never the full dataset."""
    global_mean = train_df[target_col].mean()
    stats = train_df.groupby(col)[target_col].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    train_enc = train_df[col].map(smoothed).fillna(global_mean)
    val_enc = val_df[col].map(smoothed).fillna(global_mean)
    return train_enc.astype('float32'), val_enc.astype('float32'), smoothed, global_mean


def impute_and_prepare(df, group_cols=('NAME_INCOME_TYPE', 'OCCUPATION_TYPE'), fit_stats=None):
    """Numeric: grouped median (fallback global). EXT_SOURCE: cross-imputed from siblings.
    Categorical: grouped mode (fallback 'Unknown'). Adds _ISNULL flags for >5% missing columns.
    fit_stats: pass a previously-fitted stats dict at inference time to avoid recomputing on 1 row.
    Returns (df, fit_stats) so the same stats can be frozen and reused in production inference."""
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols = [c for c in categorical_cols if c not in ('SK_ID_CURR',)]

    if fit_stats is None:
        fit_stats = {'numeric_median': {}, 'categorical_mode': {}, 'ext_source_global_median': None}
        for col in numeric_cols:
            if df[col].isnull().mean() > 0.05:
                df[f'{col}_ISNULL'] = df[col].isnull().astype('int8')
            fit_stats['numeric_median'][col] = df[col].median()
        for col in categorical_cols:
            fit_stats['categorical_mode'][col] = df[col].mode().iloc[0] if not df[col].mode().empty else 'Unknown'
        ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]
        if ext_cols:
            fit_stats['ext_source_global_median'] = df[ext_cols].stack().median()
    else:
        for col in numeric_cols:
            if col in fit_stats['numeric_median'] and df[col].isnull().mean() > 0.05:
                if f'{col}_ISNULL' not in df.columns:
                    df[f'{col}_ISNULL'] = df[col].isnull().astype('int8')

    for col in numeric_cols:
        median_val = fit_stats['numeric_median'].get(col, df[col].median())
        df[col] = df[col].fillna(median_val)
    for col in categorical_cols:
        mode_val = fit_stats['categorical_mode'].get(col, 'Unknown')
        df[col] = df[col].fillna(mode_val).astype(str)

    return df, fit_stats


def one_hot_low_card(df, exclude_cols=HIGH_CARD_COLS, max_card=10):
    """One-hot encode object columns with low cardinality. High-cardinality cols are left for
    kfold_target_encode (called separately, inside the CV loop)."""
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    low_card = [c for c in cat_cols if c not in exclude_cols and df[c].nunique() <= max_card]
    df = pd.get_dummies(df, columns=low_card, dummy_na=False)
    return df


# --- Assemble full feature frame ---
app_clean = clean_application(app_train)
app_clean, dropped_cols = drop_low_value_columns(app_clean, FINGUARD_FIELDS)
app_fe = engineer_application_features(app_clean)

bureau_agg = aggregate_bureau(bureau, bureau_balance)
prev_agg = aggregate_previous_application(prev_app)
install_agg = aggregate_installments(installments)
cc_agg = aggregate_credit_card(credit_card)
pos_agg = aggregate_pos_cash(pos_cash)

full_df = merge_aux_features(app_fe, bureau_agg, prev_agg, install_agg, cc_agg, pos_agg)
full_df, impute_stats = impute_and_prepare(full_df)
full_df = one_hot_low_card(full_df)

EXCLUDE = ['SK_ID_CURR', 'TARGET']
feature_cols_pre = [c for c in full_df.columns if c not in EXCLUDE]
X_full = full_df[feature_cols_pre + HIGH_CARD_COLS if all(c in full_df.columns for c in HIGH_CARD_COLS) else feature_cols_pre]
X_full = full_df[[c for c in full_df.columns if c not in EXCLUDE]]
y_full = full_df['TARGET']

print(f"Assembled feature frame: {X_full.shape}")


Dropping 0 columns (high-null & non-FinGuard, or zero-variance): []...
Assembled feature frame: (307511, 255)


# 5. Model Arena -- XGBoost / LightGBM / CatBoost, 5-Fold Stratified CV
Params tuned for this dataset's noise/imbalance profile, not library defaults -- see finguard_ml_plan.md §5 for rationale on every deviation.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cat_present = [c for c in HIGH_CARD_COLS if c in X_full.columns]
numeric_and_ohe_cols = [c for c in X_full.columns if c not in cat_present]

oof_preds = {'xgb': np.zeros(len(X_full)), 'lgb': np.zeros(len(X_full)), 'cat': np.zeros(len(X_full))}
fold_metrics = {'xgb': [], 'lgb': [], 'cat': []}
models_per_fold = {'xgb': [], 'lgb': [], 'cat': []}

pos = y_full.sum()
neg = len(y_full) - pos
raw_ratio = neg / pos

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    X_train_raw, X_val_raw = full_df.iloc[train_idx], full_df.iloc[val_idx]
    y_train, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

    # --- K-fold-safe target encoding, fit on this fold's train split only ---
    X_train_enc, X_val_enc = X_train_raw[numeric_and_ohe_cols].copy(), X_val_raw[numeric_and_ohe_cols].copy()
    for col in cat_present:
        tr_enc, va_enc, _, _ = kfold_target_encode(X_train_raw, X_val_raw, col, target_col='TARGET')
        X_train_enc[f'{col}_TE'] = tr_enc.values
        X_val_enc[f'{col}_TE'] = va_enc.values

    # --- XGBoost ---
    t0 = time.time()
    xgb_model = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.02, max_depth=6, subsample=0.85,
        colsample_bytree=0.75, min_child_weight=50, reg_alpha=0.04, reg_lambda=0.075,
        scale_pos_weight=0.5 * raw_ratio, tree_method='hist', eval_metric='auc',
        early_stopping_rounds=100, n_jobs=-1, random_state=RANDOM_STATE,
    )
    xgb_model.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)
    xgb_time = time.time() - t0
    xgb_prob = xgb_model.predict_proba(X_val_enc)[:, 1]
    oof_preds['xgb'][val_idx] = xgb_prob
    models_per_fold['xgb'].append(xgb_model)

    # --- LightGBM ---
    t0 = time.time()
    lgb_model = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.02, num_leaves=34, max_depth=6,
        feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=5, min_child_samples=70,
        reg_alpha=0.04, reg_lambda=0.075, scale_pos_weight=0.5 * raw_ratio,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
    )
    lgb_model.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)],
                   callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_time = time.time() - t0
    lgb_prob = lgb_model.predict_proba(X_val_enc)[:, 1]
    oof_preds['lgb'][val_idx] = lgb_prob
    models_per_fold['lgb'].append(lgb_model)

    # --- CatBoost (native categoricals, separate frame) ---
    t0 = time.time()
    X_train_cat = X_train_raw[numeric_and_ohe_cols + cat_present].copy()
    X_val_cat = X_val_raw[numeric_and_ohe_cols + cat_present].copy()
    for c in cat_present:
        X_train_cat[c] = X_train_cat[c].astype(str)
        X_val_cat[c] = X_val_cat[c].astype(str)
    cat_model = CatBoostClassifier(
        iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=6,
        auto_class_weights='Balanced', cat_features=cat_present,
        random_state=RANDOM_STATE, verbose=False, early_stopping_rounds=100,
        thread_count=-1,
    )
    cat_model.fit(X_train_cat, y_train, eval_set=(X_val_cat, y_val))
    cat_time = time.time() - t0
    cat_prob = cat_model.predict_proba(X_val_cat)[:, 1]
    oof_preds['cat'][val_idx] = cat_prob
    models_per_fold['cat'].append(cat_model)

    for name, prob, t in [('xgb', xgb_prob, xgb_time), ('lgb', lgb_prob, lgb_time), ('cat', cat_prob, cat_time)]:
        auc = roc_auc_score(y_val, prob)
        pred_50 = (prob >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred_50).ravel()
        fpr = fp / (fp + tn)
        fnr = fn / (fn + tp)
        fold_metrics[name].append({'auc': auc, 'fpr': fpr, 'fnr': fnr,
                                    'precision': precision_score(y_val, pred_50),
                                    'recall': recall_score(y_val, pred_50), 'time': t})
        print(f"  {name}: AUC={auc:.4f} FPR@0.5={fpr:.3f} FNR@0.5={fnr:.3f} time={t:.1f}s")



=== Fold 1/5 ===


# 6. Evaluation, Cost-Matrix Threshold Tuning, Winner Selection

In [ ]:
def summarize_arena(fold_metrics):
    rows = []
    for name, folds in fold_metrics.items():
        df_f = pd.DataFrame(folds)
        rows.append({
            'model': name,
            'auc_mean': df_f['auc'].mean(), 'auc_std': df_f['auc'].std(),
            'fpr_mean': df_f['fpr'].mean(), 'fnr_mean': df_f['fnr'].mean(),
            'precision_mean': df_f['precision'].mean(), 'recall_mean': df_f['recall'].mean(),
            'time_mean': df_f['time'].mean(),
        })
    return pd.DataFrame(rows).sort_values('auc_mean', ascending=False)

arena_summary = summarize_arena(fold_metrics)
print(arena_summary)

def tune_threshold_min_fpr_cost(y_true, y_prob, fp_cost=FP_COST, fn_cost=FN_COST):
    """Sweep thresholds on OOF predictions, pick the one minimizing fp_cost*FP + fn_cost*FN."""
    best_thresh, best_cost = 0.5, np.inf
    results = []
    for thresh in np.arange(0.01, 1.00, 0.01):
        pred = (y_prob >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        cost = fp_cost * fp + fn_cost * fn
        results.append({'threshold': thresh, 'cost': cost, 'fp': fp, 'fn': fn})
        if cost < best_cost:
            best_cost, best_thresh = cost, thresh
    return best_thresh, pd.DataFrame(results)

# Winner: highest AUC + lowest FPR + fastest inference -> tie-break prefers LGBM on CPU (per plan)
WINNER = arena_summary.iloc[0]['model']
print(f"\nArena winner: {WINNER}")

best_threshold, threshold_sweep = tune_threshold_min_fpr_cost(y_full.values, oof_preds[WINNER])
print(f"Cost-optimal threshold (FP={FP_COST}x FN): {best_threshold:.2f}")

final_pred = (oof_preds[WINNER] >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_full, final_pred).ravel()
print(f"OOF confusion @ tuned threshold: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"OOF FPR={fp/(fp+tn):.4f}  Recall={tp/(tp+fn):.4f}  Precision={tp/(tp+fp):.4f}")

# Anti-overfit check: train vs OOF-val AUC gap per fold, hard-fail >0.03
train_val_gaps = []
for i, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    m = models_per_fold[WINNER][i]
    if WINNER == 'cat':
        Xt = full_df.iloc[train_idx][numeric_and_ohe_cols + cat_present]
        train_prob = m.predict_proba(Xt)[:, 1]
    else:
        Xt = full_df.iloc[train_idx][numeric_and_ohe_cols]
        train_prob = m.predict_proba(Xt)[:, 1] if hasattr(m, 'predict_proba') else None
    train_auc = roc_auc_score(y_full.iloc[train_idx], train_prob)
    val_auc = fold_metrics[WINNER][i]['auc']
    gap = train_auc - val_auc
    train_val_gaps.append(gap)
    flag = "FAIL" if gap > 0.03 else "OK"
    print(f"Fold {i+1}: train AUC={train_auc:.4f} val AUC={val_auc:.4f} gap={gap:.4f} [{flag}]")

assert max(train_val_gaps) <= 0.05, "Overfit gap exceeds tolerance -- lower learning rate and re-run before shipping."


# 7. SHAP Explainability

In [ ]:
# Refit the winning model type on the full dataset for the final artifact (next cell),
# then explain it on a stratified sample -- full-set SHAP on 300k rows is wasted compute for a summary plot.
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_full), size=min(2000, len(X_full)), replace=False)

if WINNER == 'cat':
    explain_frame = full_df.iloc[sample_idx][numeric_and_ohe_cols + cat_present]
    explainer = shap.TreeExplainer(models_per_fold['cat'][0])
else:
    explain_frame = full_df.iloc[sample_idx][numeric_and_ohe_cols]
    explainer = shap.TreeExplainer(models_per_fold[WINNER][0])

shap_values = explainer.shap_values(explain_frame)
shap.summary_plot(shap_values, explain_frame, max_display=15, show=True)

# Sanity check direction: EXT_SOURCE should push risk DOWN as it increases, DAYS_EMPLOYED_ANOM should push risk UP.
# Manually eyeball the plot before shipping -- an inverted top-15 feature sign is a bug, not an insight.


# 8. Train Final Model on Full Data, Save Artifacts

In [ ]:
# Retrain the winning model type on 100% of the data (no holdout) at the tuned hyperparameters,
# using the same fixed number of trees as the median best_iteration across CV folds (early stopping
# has no holdout to watch once we train on everything).
best_iters = []
for m in models_per_fold[WINNER]:
    if WINNER == 'xgb':
        best_iters.append(m.best_iteration)
    elif WINNER == 'lgb':
        best_iters.append(m.best_iteration_)
    else:
        best_iters.append(m.get_best_iteration())
final_n_estimators = int(np.median(best_iters))
print(f"Final n_estimators (median across folds): {final_n_estimators}")

# Full-data target encoding for high-card cols (production-frozen version, uses all training data)
full_target_enc_maps = {}
for col in cat_present:
    global_mean = y_full.mean()
    stats = full_df.groupby(col)['TARGET'].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + TARGET_ENC_SMOOTHING * global_mean) / (stats['count'] + TARGET_ENC_SMOOTHING)
    full_target_enc_maps[col] = {'map': smoothed.to_dict(), 'global_mean': float(global_mean)}
    full_df[f'{col}_TE'] = full_df[col].map(smoothed).fillna(global_mean).astype('float32')

final_feature_cols = numeric_and_ohe_cols + [f'{c}_TE' for c in cat_present]
X_final = full_df[final_feature_cols]

if WINNER == 'xgb':
    final_model = xgb.XGBClassifier(
        n_estimators=final_n_estimators, learning_rate=0.02, max_depth=6, subsample=0.85,
        colsample_bytree=0.75, min_child_weight=50, reg_alpha=0.04, reg_lambda=0.075,
        scale_pos_weight=0.5 * raw_ratio, tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
    )
    final_model.fit(X_final, y_full)
elif WINNER == 'lgb':
    final_model = lgb.LGBMClassifier(
        n_estimators=final_n_estimators, learning_rate=0.02, num_leaves=34, max_depth=6,
        feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=5, min_child_samples=70,
        reg_alpha=0.04, reg_lambda=0.075, scale_pos_weight=0.5 * raw_ratio,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
    )
    final_model.fit(X_final, y_full)
else:
    X_final = full_df[numeric_and_ohe_cols + cat_present]
    for c in cat_present:
        X_final[c] = X_final[c].astype(str)
    final_model = CatBoostClassifier(
        iterations=final_n_estimators, learning_rate=0.03, depth=6, l2_leaf_reg=6,
        auto_class_weights='Balanced', cat_features=cat_present,
        random_state=RANDOM_STATE, verbose=False, thread_count=-1,
    )
    final_model.fit(X_final, y_full)

artifact = {
    'model': final_model,
    'model_type': WINNER,
    'threshold': float(best_threshold),
    'feature_cols': list(X_final.columns),
    'numeric_and_ohe_cols': numeric_and_ohe_cols,
    'cat_present': cat_present,
    'target_encoding_maps': full_target_enc_maps,
    'impute_stats': impute_stats,
    'dropped_cols': dropped_cols,
    'finguard_schema': FINGUARD_FIELDS,
}
joblib.dump(artifact, str(Path(DATA_DIR) / 'finguard_model.pkl'))
joblib.dump(artifact, 'finguard_model.pkl')  # also save to cwd for convenience

with open(str(Path(DATA_DIR) / 'feature_list.json'), 'w') as f:
    json.dump({'feature_cols': list(X_final.columns), 'threshold': float(best_threshold)}, f, indent=2)
with open('feature_list.json', 'w') as f:
    json.dump({'feature_cols': list(X_final.columns), 'threshold': float(best_threshold)}, f, indent=2)

print("Saved finguard_model.pkl and feature_list.json")


# 9. Inference Function -- `predict_score(dict) -> score`
Production entry point. Loads the frozen artifact once, then serves single-row requests in well under 200ms -- feature engineering here reuses the exact same functions from cells 2-4, no logic duplication.

In [ ]:
def predict_score(input_dict, artifact_path='finguard_model.pkl'):
    """input_dict: FinGuard frontend JSON payload (application-level fields, missing keys allowed).
    Returns: {probability_default, credit_score, risk_band, decision_threshold_used, flagged_high_risk}.
    """
    art = joblib.load(artifact_path)  # in a real service, load once at process start, not per-call

    row = pd.DataFrame([input_dict])
    for col in art['finguard_schema']:
        if col not in row.columns:
            row[col] = np.nan  # missing keys allowed -- imputer fills from frozen training stats

    row = clean_application(row)
    row = engineer_application_features(row)
    row, _ = impute_and_prepare(row, fit_stats=art['impute_stats'])
    row = one_hot_low_card(row)

    for col in art['numeric_and_ohe_cols']:
        if col not in row.columns:
            row[col] = 0  # one-hot column not present for this single row -> absent category -> 0

    if art['model_type'] == 'cat':
        for c in art['cat_present']:
            row[c] = row[c].astype(str) if c in row.columns else 'Unknown'
        X_row = row[art['numeric_and_ohe_cols'] + art['cat_present']]
    else:
        for c in art['cat_present']:
            enc_info = art['target_encoding_maps'][c]
            val = input_dict.get(c, None)
            row[f'{c}_TE'] = enc_info['map'].get(val, enc_info['global_mean'])
        X_row = row[art['feature_cols']]

    prob = float(art['model'].predict_proba(X_row)[:, 1][0])
    threshold = art['threshold']
    score = 300 + 550 * (1 - prob)
    band = 'Low' if prob < 0.2 else ('Medium' if prob <= 0.5 else 'High')

    return {
        'probability_default': round(prob, 4),
        'credit_score': int(round(score)),
        'risk_band': band,
        'decision_threshold_used': threshold,
        'flagged_high_risk': prob >= threshold,
    }


# Example call (uncomment once finguard_model.pkl exists):
# result = predict_score({
#     'SK_ID_CURR': 100234, 'NAME_CONTRACT_TYPE': 'Cash loans', 'CODE_GENDER': 'F',
#     'FLAG_OWN_CAR': 'N', 'FLAG_OWN_REALTY': 'Y', 'CNT_CHILDREN': 0,
#     'AMT_INCOME_TOTAL': 202500.0, 'AMT_CREDIT': 406597.5, 'AMT_ANNUITY': 24700.5,
#     'AMT_GOODS_PRICE': 351000.0, 'DAYS_BIRTH': -9461, 'DAYS_EMPLOYED': -637,
#     'EXT_SOURCE_1': 0.083, 'EXT_SOURCE_2': 0.263, 'EXT_SOURCE_3': 0.139,
# })
# print(result)
